# Pose Training — 5KP No Roof Apex
Jalankan cell dari atas ke bawah secara berurutan.

> **Pastikan runtime sudah T4 GPU:** Runtime → Change runtime type → T4 GPU

## Cell 1 — Clone repo & install dependencies

In [ ]:
import os

REPO_DIR = "/content/sdi-helper"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/riorus94/sdi-helper.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install -e . --quiet
!pip install "numpy>=2.0,<2.1" --quiet --upgrade

## Cell 2 — Cek jumlah JSON annotation & source image tersedia
Catatan: konversi label bisa tetap jalan meski image kosong, **tapi training tidak bisa**. Jika `source images` = 0, upload image dulu sebelum Cell 5.

In [ ]:
from pathlib import Path

json_src_dir = Path("yolo_training/side_view_dataset/labelme_json_stanford_screening")
legacy_json_dir = Path("yolo_training/side_view_dataset/labelme_json")
img_dir = Path("dataset_raw/images/train/side")

screening_count = len(list(json_src_dir.glob('*.json')))
legacy_count = len(list(legacy_json_dir.glob('*.json')))

print(f"json_stanford_screening : {screening_count} files")
print(f"json_legacy_ignored     : {legacy_count} files")
print(f"source images            : {len(list(img_dir.glob('*.*')))} files")

# Training hanya boleh memakai JSON screening hasil Agent 1 CLIP + ground_ref fix.
# Jangan fallback ke labelme_json legacy, karena bisa berisi ground_ref/front-rear lama.
JSON_SRC = str(json_src_dir) if screening_count > 0 else None

print(f"\nJSON source yang akan dipakai: {JSON_SRC}")
if JSON_SRC is None and legacy_count > 0:
    print("⚠️ labelme_json legacy ditemukan tapi sengaja diabaikan. Upload/fix JSON ke labelme_json_stanford_screening dulu.")

## Cell 3 — Konversi JSON → YOLO pose labels
Lewati cell ini kalau `JSON_SRC` di atas adalah `None`.

In [ ]:
assert JSON_SRC is not None, "Tidak ada JSON annotation. Upload dulu file JSON ke folder yolo_training/side_view_dataset/labelme_json_stanford_screening/"

!python yolo_training/labelme_to_yolo_pose.py \
    --input "{JSON_SRC}" \
    --output yolo_training/side_view_dataset/labels_pose_5kp_no_roof \
    --img-dir dataset_raw/images/train/side \
    --keypoints ground_ref,front_wheel_center,front_wheel_ground,rear_wheel_center,rear_wheel_ground

## Cell 4 — Cek hasil konversi

In [ ]:
from pathlib import Path

labels_dir = Path("yolo_training/side_view_dataset/labels_pose_5kp_no_roof")
txt_count = len(list(labels_dir.glob("*.txt")))
print(f"Pose label txt files: {txt_count}")

if txt_count == 0:
    print("\n⚠ STOP: Konversi menghasilkan 0 label.")
    print("Kemungkinan penyebab: imagePath di JSON tidak cocok dengan nama file di dataset_raw/images/train/side/")
    print("Cek nama file JSON pertama:")
    import json
    first_json = sorted(Path(JSON_SRC).glob("*.json"))[0]
    data = json.loads(first_json.read_text())
    print(f"  imagePath di JSON  : {data.get('imagePath')}")
    print(f"  JSON filename       : {first_json.name}")
else:
    print(f"✅ Siap training dengan {txt_count} labeled images")

## Cell 5 — Jalankan pose training
Hanya jalankan kalau Cell 4 menunjukkan `✅`.

In [ ]:
import os
from pathlib import Path

# Training tetap butuh image source (meskipun konversi label bisa fallback ke dimensi JSON)
img_dir = Path("dataset_raw/images/train/side")
img_count = len(list(img_dir.glob("*.*")))
assert img_count > 0, (
    "Source images masih 0. Upload image dulu ke dataset_raw/images/train/side "
    "(atau mount Google Drive lalu copy ke folder itu), baru jalankan training."
)

os.environ["POSE_KEYPOINTS"] = "ground_ref,front_wheel_center,front_wheel_ground,rear_wheel_center,rear_wheel_ground"
os.environ["POSE_LABEL_DIR"] = "yolo_training/side_view_dataset/labels_pose_5kp_no_roof"
os.environ["POSE_ALLOW_LEGACY_SOURCE"] = "1"
os.environ["POSE_REQUIRE_WHEEL_IMAGES"] = "0"
os.environ["POSE_MODEL_WEIGHTS"] = "yolov8s-pose.pt"
os.environ["POSE_EPOCHS"] = "120"
os.environ["POSE_IMGSZ"] = "768"
os.environ["POSE_BATCH"] = "8"
os.environ["POSE_LR0"] = "0.001"
os.environ["POSE_WARMUP_EPOCHS"] = "3"
os.environ["POSE_PATIENCE"] = "60"
os.environ["POSE_MOSAIC"] = "0.0"
os.environ["POSE_SCALE"] = "0.10"
os.environ["POSE_TRANSLATE"] = "0.03"
os.environ["POSE_RUN_NAME"] = "colab_pose_5kp_no_roof_tuned"

!python yolo_training/train_pose.py

## Cell 6 — Cek hasil dan download best.pt

In [ ]:
from pathlib import Path

best = Path("yolo_training/runs/colab_pose_5kp_no_roof_tuned/weights/best.pt")
if best.exists():
    size_mb = best.stat().st_size / 1024 / 1024
    print(f"✅ best.pt tersedia: {best} ({size_mb:.1f} MB)")
    from google.colab import files
    files.download(str(best))
else:
    print("❌ best.pt tidak ditemukan, cek log training di atas")